# Adult Chest X-Ray Model — Pre-trained (No Training Needed!)

This notebook uses a **pre-trained DenseNet121** from the [TorchXRayVision](https://github.com/mlmed/torchxrayvision) library.

The model was trained on the **RSNA Pneumonia Detection Challenge** dataset (adult chest X-rays).

**No training, no dataset download, no GPU needed!**

We just:
1. Install the library
2. Load the pre-trained model
3. Test it on sample images
4. Export the model for use in our Streamlit app

## Step 1: Install Dependencies

In [ ]:
!pip install -q torchxrayvision scikit-image

## Step 2: Load Pre-trained Model

In [ ]:
import torchxrayvision as xrv
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import skimage
import os

# Load the RSNA pre-trained model (auto-downloads ~27MB)
print("Loading pre-trained model...")
model = xrv.models.DenseNet(weights="densenet121-res224-rsna")
model.eval()

print(f"\n✅ Model loaded!")
print(f"Architecture: DenseNet121")
print(f"Trained on: RSNA Pneumonia Detection Challenge (adult X-rays)")
print(f"\nOutput pathologies ({len(model.pathologies)}):")
for i, p in enumerate(model.pathologies):
    marker = ' ← WE USE THIS' if p == 'Pneumonia' else ''
    print(f"  [{i:2d}] {p}{marker}")

## Step 3: Test on a Sample Image
Upload a chest X-ray image to test, or use a generated sample.

In [ ]:
from google.colab import files

def preprocess_for_xrv(image_path):
    """Preprocess an image for TorchXRayVision model."""
    # Read image
    img = skimage.io.imread(image_path)
    
    # Convert to grayscale if needed
    if len(img.shape) == 3:
        img = img.mean(axis=2)  # Average RGB channels
    
    # Normalize to [0, 255] range
    img = img.astype(np.float32)
    if img.max() > 0:
        img = (img - img.min()) / (img.max() - img.min()) * 255.0
    
    # Resize to 224x224
    img = skimage.transform.resize(img, (224, 224), anti_aliasing=True)
    img = img * 255.0  # skimage resize normalizes to [0,1]
    
    # XRV expects values normalized to [-1024, 1024]
    img = xrv.datasets.normalize(img, maxval=255, reshape=True)
    
    # Add batch dimension
    img_tensor = torch.from_numpy(img).unsqueeze(0)
    return img_tensor

def predict(model, image_path):
    """Run prediction on a single image."""
    img_tensor = preprocess_for_xrv(image_path)
    
    with torch.no_grad():
        output = model(img_tensor)
    
    # Get pneumonia probability
    pneumonia_idx = list(model.pathologies).index('Pneumonia')
    pneumonia_prob = torch.sigmoid(output[0, pneumonia_idx]).item()
    
    # Display results
    img = skimage.io.imread(image_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    ax1.imshow(img, cmap='gray' if len(img.shape) == 2 else None)
    ax1.set_title('Input X-Ray')
    ax1.axis('off')
    
    # Show top pathology scores
    scores = torch.sigmoid(output[0]).numpy()
    top_indices = np.argsort(scores)[::-1][:8]
    top_names = [model.pathologies[i] for i in top_indices]
    top_scores = [scores[i] for i in top_indices]
    colors = ['red' if n == 'Pneumonia' else 'steelblue' for n in top_names]
    
    ax2.barh(range(len(top_names)), top_scores, color=colors)
    ax2.set_yticks(range(len(top_names)))
    ax2.set_yticklabels(top_names)
    ax2.set_xlabel('Probability')
    ax2.set_title('Top Pathology Scores')
    ax2.set_xlim(0, 1)
    ax2.invert_yaxis()
    
    # Verdict
    verdict = 'PNEUMONIA DETECTED' if pneumonia_prob > 0.5 else 'NORMAL'
    fig.suptitle(f'Result: {verdict} (Pneumonia: {pneumonia_prob:.1%})', 
                 fontsize=14, fontweight='bold',
                 color='red' if pneumonia_prob > 0.5 else 'green')
    plt.tight_layout()
    plt.show()
    
    return pneumonia_prob

# Upload and test
print("Upload a chest X-ray image to test:")
try:
    uploaded = files.upload()
    for filename in uploaded.keys():
        print(f"\nAnalyzing: {filename}")
        prob = predict(model, filename)
        print(f"Pneumonia probability: {prob:.4f}")
except Exception as e:
    print(f"Upload skipped or failed: {e}")
    print("You can also test manually by placing an image in /content/")

## Step 4: Export Model for Streamlit App
Save the model weights so they can be used in `app.py`.

In [ ]:
# Save model for the Streamlit app
save_path = '/content/densenet121_adult_rsna.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'pathologies': list(model.pathologies),
    'weights_name': 'densenet121-res224-rsna',
    'model_type': 'torchxrayvision',
    'input_format': 'grayscale_224x224_normalized_neg1024_to_1024',
}, save_path)

# Check file size
size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"✅ Model saved to: {save_path}")
print(f"   File size: {size_mb:.1f} MB")
print(f"\n📥 Download this file from the Files sidebar.")
print(f"   Place it in your project folder alongside densenet121_pneumonia.pth")

## Step 5: Model Information
Print model details for documentation.

In [ ]:
# Model summary for documentation
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 50)
print("  Adult Model — Summary")
print("=" * 50)
print(f"  Library:        TorchXRayVision")
print(f"  Architecture:   DenseNet121")
print(f"  Weights:        densenet121-res224-rsna")
print(f"  Training Data:  RSNA Pneumonia Detection Challenge")
print(f"  Population:     Adult patients")
print(f"  Parameters:     {total_params:,}")
print(f"  Input:          224x224 grayscale")
print(f"  Output:         {len(model.pathologies)} pathologies")
print(f"  Key Output:     Pneumonia probability")
print(f"")
print(f"  Pathologies detected:")
for p in model.pathologies:
    print(f"    - {p}")

## Summary

✅ **No training was needed!** We used a pre-trained model from TorchXRayVision.

**What we did:**
1. Installed `torchxrayvision` (1 line)
2. Loaded pre-trained RSNA model (~27MB auto-download)
3. Tested on a chest X-ray
4. Exported model weights for the Streamlit app

**Next Steps:**
1. Download `densenet121_adult_rsna.pth` from the Files sidebar
2. Place it in your project folder
3. The updated `app.py` will auto-detect both models and let users choose